In [1]:
import nltk

nltk.data.show_cfg('grammars/book_grammars/sql0.fcfg')

% start S
S[SEM=(?np + WHERE + ?vp)] -> NP[SEM=?np] VP[SEM=?vp]
VP[SEM=(?v + ?pp)] -> IV[SEM=?v] PP[SEM=?pp]
VP[SEM=(?v + ?ap)] -> IV[SEM=?v] AP[SEM=?ap]
NP[SEM=(?det + ?n)] -> Det[SEM=?det] N[SEM=?n]
PP[SEM=(?p + ?np)] -> P[SEM=?p] NP[SEM=?np]
AP[SEM=?pp] -> A[SEM=?a] PP[SEM=?pp]
NP[SEM='Country="greece"'] -> 'Greece'
NP[SEM='Country="china"'] -> 'China'
Det[SEM='SELECT'] -> 'Which' | 'What'
N[SEM='City FROM city_table'] -> 'cities'
IV[SEM=''] -> 'are'
A[SEM=''] -> 'located'
P[SEM=''] -> 'in'


In [2]:
from nltk import load_parser
cp = load_parser('grammars/book_grammars/sql0.fcfg')
query = 'What cities are located in Greece'
trees = list(cp.parse(query.split()))
answer = trees[0].label()['SEM']
answer = [s for s in answer if s]
q = ' '.join(answer)
print(q)

SELECT City FROM city_table WHERE Country="greece"


In [3]:
from nltk.sem import chat80
rows = chat80.sql_query('corpora/city_database/city.db', q)
for r in rows: print(r[0], end=" ")

athens 

In [4]:
gram = nltk.grammar.FeatureGrammar.fromstring("""
% start S
S[SEM=(?np + WHERE + ?vp)] -> NP[SEM=?np] VP[SEM=?vp]
VP[SEM=(?v + ?pp)] -> IV[SEM=?v] PP[SEM=?pp]
VP[SEM=(?v + ?ap)] -> IV[SEM=?v] AP[SEM=?ap]
NP[SEM=(?det + City + ?n)] -> Det[AGR= 'w',SEM=?det] N[WC= 'City', SEM=?n]
NP[SEM=(?det + Country + ?n)] -> Det[AGR= 'w',SEM=?det] N[WC= 'Country', SEM=?n]
NP[SEM=(?det + ?a + ?n)] -> Det[AGR= 'h', SEM=?det] A[SEM=?a] N[SEM=?n]
PP[SEM=(?p + ?np)] -> P[SEM=?p] NP[SEM=?np]
PP[SEM=(?np + ?p)] -> NP[SEM=?np] P[SEM=?p]
PP[SEM=(?p)] -> P[SEM=?p]
 
AP[SEM=?pp] -> A[SEM=?a] PP[SEM=?pp]
AP[SEM=(?np +?ap)] -> NP[SEM =?np] AP[SEM=?ap]
NP[SEM='Country="greece"'] -> 'Greece'
NP[SEM='Country="china"'] -> 'China'
NP[SEM='City= "athens"'] -> 'Athens'
Det[AGR ='w', SEM='SELECT'] -> 'Which' | 'What' 
Det[AGR ='h', SEM='SELECT'] -> 'How'
N[WC ='City', SEM='FROM city_table'] -> 'cities'
N[WC ='Country', SEM='FROM city_table'] -> 'country'
A[SEM=' count(*)'] -> 'many'
IV[SEM=''] -> 'are'|'is'
A[SEM=''] -> 'located'
P[SEM=''] -> 'in'
 
""")

In [5]:
#query = 'Which cities are located in Greece'
#query = 'How many cities are located in Greece'
query = 'What country is Athens located in'

parser = nltk.parse.FeatureEarleyChartParser(gram)
trees = list(parser.parse(query.split()))
answer = trees[0].label()['SEM']
answer = [s for s in answer if s]
q = ' '.join(answer)
print(q)


SELECT Country FROM city_table WHERE City= "athens"


In [6]:
read_expr = nltk.sem.Expression.fromstring

In [7]:
read_expr('P & Q')

<AndExpression (P & Q)>

In [8]:
read_expr('-(P & Q)')

<NegatedExpression -(P & Q)>

In [9]:
read_expr('P | (R -> Q)')

<OrExpression (P | (R -> Q))>

In [10]:
read_expr('P <-> -- P')

<IffExpression (P <-> --P)>

In [11]:
NotFnS = read_expr('-north_of(f, s)') 
SnF = read_expr('north_of(s, f)') 
R = read_expr('all x. all y. (north_of(x, y) -> -north_of(y, x))')
prover = nltk.Prover9()
prover.prove(NotFnS, [SnF, R])

LookupError: 

===========================================================================
NLTK was unable to find the prover9 file!
Use software specific configuration parameters or set the PROVER9 environment variable.

  Searched in:
    - /usr/local/bin/prover9
    - /usr/local/bin/prover9/bin
    - /usr/local/bin
    - /usr/bin
    - /usr/local/prover9
    - /usr/local/share/prover9

  For more information on prover9, see:
    <https://www.cs.unm.edu/~mccune/prover9/>
===========================================================================